# ML-02 — Research Question and Provisional Lane

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.**

I want to investigate which pages should be reviewed first for a possible refresh, expansion, protection, pruning, or monitoring action. This lane fits the starter dataset because it contains page-level search, traffic, content-age, position, CTR, and engagement signals, and the starter pipeline already demonstrates that a learned ranking can improve the order in which pages are reviewed. I am choosing this as a provisional lane rather than assuming that a model can prove which pages will benefit from a refresh. I can change the lane later if the fuller data shows a more useful question.


In [4]:
from pathlib import Path
import pandas as pd

# Make the notebook work whether it is launched from the repo root,
# work/notebooks/, or another nearby working directory.
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd.parent.parent]
data_path = next(
    (p / 'data' / 'raw' / 'content_refresh_anonymized.csv'
     for p in candidates
     if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists()),
    None,
)

if data_path is None:
    raise FileNotFoundError('Could not find data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(data_path)
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')


Rows: 30,000
Columns: 44


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given the observable search and content signals available before review, which pages should a limited content team review first for a possible refresh or another appropriate action?

**Decision:** Decide which pages belong near the top of a human review queue.

**Unit of analysis:** One page/content item for a client.

**Output:** A ranked list of candidate pages with a score, an appropriate suggested action, and reason codes explaining the evidence behind the ranking.

**Action:** A content or SEO reviewer could inspect the highest-ranked pages first and decide whether to refresh, expand, protect, prune, or simply monitor them. The system is decision support; the final action remains with a human.

**Cost of a wrong recommendation:** Ranking a weak candidate too highly can waste limited editorial/SEO time on a page that does not need attention. Ranking a genuinely important opportunity too low can delay useful work and leave potential search visibility or engagement improvements unaddressed. The relative cost depends on the page and the action, so I will avoid treating every error as equally expensive.

**Why data/ML can help:** The dataset contains multiple observable signals about visibility, traffic, position, CTR, content age, freshness, and engagement. A model can combine these signals consistently and rank many pages, while a simple baseline provides a transparent comparison. The goal is not simply to train a model; it is to improve a constrained human review decision and explain why pages were surfaced.


In [5]:
# A few supporting measurements from the actual starter data.
required = ['trend_direction', 'impressions_90d', 'content_age_days']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

declining_share = (df['trend_direction'].eq('down').mean() * 100)
visible_500 = (df['impressions_90d'].ge(500).mean() * 100)
median_impressions = df['impressions_90d'].median()

print(f"Pages marked 'down' in the starter's current-window proxy: {declining_share:.1f}%")
print(f"Pages with at least 500 impressions in 90 days: {visible_500:.1f}%")
print(f"Median 90-day impressions: {median_impressions:,.0f}")


Pages marked 'down' in the starter's current-window proxy: 54.2%
Pages with at least 500 impressions in 90 days: 55.8%
Median 90-day impressions: 731


## 3. Quick look at the data (2-3 real numbers)

The starter dataset contains **30,000 pseudonymized page rows and 44 columns**. The code above also measures the share currently marked `down`, the share with at least 500 impressions in 90 days, and the median 90-day impressions directly from the CSV.

These measurements make the lane worth investigating because there is enough page-level variation to define a review queue, while the dataset contains several different signals that can be combined rather than relying on one rule. The starter results also provide an initial reference point: the documented baseline Precision@50 is 0.240, while the shipped random-forest result is about 0.740 on this starter slice, although the exact model number can vary with library versions. This is evidence that ranking deserves further investigation, not proof that the same performance will hold on the full warehouse.


## 4. Careful words: what I can and can't claim

I can investigate whether observable signals are associated with current or future search/content outcomes, and I can evaluate whether a ranking method helps prioritize a human review queue under a defined validation setup. I can describe the result as **observed, measured, directional, or decision-support**.

I cannot claim that a high score guarantees that a page will recover after a refresh, that a refresh caused a recovery, or that the model has discovered a Google ranking factor. The starter `trend_direction` label is a current-window proxy rather than an ideal future outcome. For the eventual capstone, I should prefer a leakage-safe future outcome, such as using a prior feature window to define what happens in a later window. I also need to check consolidation, seasonality, low-volume noise, and leakage before making stronger recommendations.


In [6]:
# Self-check: verify that the fields used for this framing are present.
print('Required framing fields are present:', all(c in df.columns for c in required))
print('Starter data loaded successfully:', df.shape)


Required framing fields are present: True
Starter data loaded successfully: (30000, 44)


## Self-check

- [x] I selected a provisional predefined lane.
- [x] I named the decision, action, unit of analysis, output, and cost of a wrong recommendation.
- [x] I loaded the starter CSV and calculated supporting numbers from it.
- [x] I am treating the starter decline label as a proxy, not as a future causal outcome.
- [x] I am framing this as decision support, not as a guarantee or a claim about Google's algorithm.
- [ ] Run all cells and inspect the outputs before committing.
